In [13]:
! pip install bs4 # in case you don't have it installed
! pip install contractions

# Dataset: https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Beauty_v1_00.tsv.gz
#          https://web.archive.org/web/20201127142707if_/https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Office_Products_v1_00.tsv.gz


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [contractions] [anyascii]

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [23]:
import pandas as pd
import numpy as np
import nltk
import csv
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')
import re
from bs4 import BeautifulSoup
import contractions
 

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Dataset Preparation

## Read Data

In [3]:
raw_data = pd.read_csv('./data.tsv', sep='\t', quoting=csv.QUOTE_NONE)

## Keep Reviews and Ratings

*Sample reviews and ratings*

In [4]:
data = raw_data.loc[:, ['review_body', 'star_rating']]
data.sample(3, random_state=42)

,review_body,star_rating
267839,GREAT!!!!,5
531637,Awfully expensive for a cardboard box and asse...,3
1171617,What a great deal all items were packaged well...,5


*Rating statistics*

In [5]:
data.describe()

,star_rating
count,2.642434e+06
mean,4.072539e+00
std,1.386968e+00
min,1.000000e+00
25%,4.000000e+00
50%,5.000000e+00
75%,5.000000e+00
max,5.000000e+00


*Count of each type of rating*

In [6]:
data['star_rating'].value_counts()

star_rating
5    1584192
4     418694
1     307234
3     193818
2     138496
Name: count, dtype: int64

 ## Relabeling and Sampling
 
First form three classes and print their statistics. Then randomly select 100,000 reviews from the positive and 100,000 reviews from the negative



In [7]:
def transform(row):
    rating = row['star_rating']
    if rating < 3:
        return 0
    elif rating > 3:
        return 1
    else:
        return 0.5

data['sentiment'] = data.apply(transform, axis=1)

In [8]:
data['sentiment'].value_counts()
# print py

sentiment
1.0    2002886
0.0     445730
0.5     193818
Name: count, dtype: int64

In [9]:
positive_reviews = data[data['sentiment'] == 1].sample(100000, random_state=42)
negative_reviews = data[data['sentiment'] == 0].sample(100000, random_state=42)
data_downsized = pd.concat([positive_reviews, negative_reviews])

# Data Cleaning



In [15]:
def clean_review(row):
    review = str(row['review_body'])
    text = review.lower()
    text = BeautifulSoup(text, "html.parser").get_text(strip=True)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = contractions.fix(text)
    return text

data_downsized['review'] = data_downsized.apply(clean_review, axis=1)

/tmp/ipykernel_1697/1833853279.py:4: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  text = BeautifulSoup(text, "html.parser").get_text(strip=True)


In [ ]:
# Print in py file
data_cleaned = data_downsized.loc[:, ['review', 'sentiment']]
print(f"Average review length before cleaning: {data_downsized['review_body'].astype(str).str.len().mean()}")
print(f"Average review length after cleaning: {data_cleaned['review'].apply(len).mean()}")

Average review length before cleaning: 318.6051992079683
Average review length after cleaning: 302.265975


# Pre-processing

## remove the stop words 

In [25]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def remove_stopwords(row):
    text = row['review']
    words = word_tokenize(text)
    stop = set(stopwords.words('english'))
    return " ".join([w for w in words if w not in stop])

data_cleaned['review'] = data_cleaned.apply(remove_stopwords, axis=1)

## perform lemmatization  

In [26]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize(row):
    text = row['review']
    words = word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(w) for w in words])

data_cleaned['review'] = data_cleaned.apply(lemmatize, axis=1)

In [44]:
data_cleaned

,review,sentiment
1707224,really like organizer right size compartment t...,1.0
1205552,binder like got high school continue useful ye...,1.0
455154,good quality except metal bar wide hangar,1.0
2373664,using paper year learned long time ago use pap...,1.0
2413772,use portable recorder tape listen tape walk ne...,1.0
...,...,...
1705693,bought bag sale target wanted throwaway pen ar...,0.0
1934872,love cross pen find ink replacement anywhere d...,0.0
2124160,returned many character word required mu ha ha...,0.0
1264791,product worked great beginning would hold char...,0.0


In [45]:
showcase = pd.DataFrame()
showcase['pre-cleaning'] = data_downsized.head(3)['review_body']
showcase['post-processing'] = data_cleaned.head(3)['review']
showcase

,pre-cleaning,post-processing
1707224,I really like this organizer. It has all the ...,really like organizer right size compartment t...
1205552,"Binders like this got me through high school, ...",binder like got high school continue useful ye...
455154,Good quality except the metal bar is too wide ...,good quality except metal bar wide hangar


In [ ]:
# Print in py file
print(f"Average review length before cleaning and pre-processing: {data_downsized['review_body'].astype(str).str.len().mean()}")
print(f"Average review length after pre-processing: {data_cleaned['review'].apply(len).mean()}")

Average review length before cleaning and pre-processing: 318.6051992079683
Average review length after pre-processing: 187.935225


# Bigram Feature Extraction

# Perceptron

# SVM

# Logistic Regression

# Naive Bayes